# New Contributor Onboarding Funnel — LKML5Ws

> *"This analysis focuses on *netdev* (a core kernel mailing list) as an initial case study."*

---

### Funnel Stages
```
New Hash  →  First Patch  →  First Review (Reviewed-by / Acked-by)  →  Retention (6 / 12 months)
```

### Extracted Metrics
| Metric | Description |
|--------|-------------|
| **Mean Time to First Patch** | Average days between first message and first patch submission |
| **Review Conversion Rate** | % of newcomers who receive ≥1 `Reviewed-by` or `Acked-by` tag |
| **Cohort Retention Rate** | % still active after 6 / 12 months, grouped by entry year |

### Notebook Structure
1. Setup & Config
2. Data Loading
3. Stage 1 — New Contributor Detection
4. Stage 2 — First Patch Detection
5. Stage 3 — First Review Tag Detection
6. Stage 4 — Retention Analysis
7. Derived Metrics
8. Export for Visualization
9. Download Results to Your Computer

## 1. Setup & Config

As this project originated in a Colab Notebook, mounting the drive via this section was essential.

In [ ]:
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import BooleanType
from datetime import datetime
import pandas as pd
import numpy as np
import re
import json
import os

spark = SparkSession.builder \
    .appName("LKML_Onboarding_Funnel") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# ── Load configuration from input.json ─────────────────────────────────────
CONFIG_PATH = "input.json"

# Auto-create default config if the file doesn't exist yet
if not os.path.exists(CONFIG_PATH):
    default_config = {
        "base_dir": "./data",
        "list_name": "org.kernel.vger.netdev",
        "analysis_years": {"start": 2004, "end": 2025},
        "retention_months": [6, 12],
        "regex": {
            "patch_subject": "(?i)\\[patch",
            "review_body": "(?i)(Reviewed-by|Acked-by)\\s*:"
        },
        "columns": {
            "from_col": "from",
            "subject_col": "subject",
            "body_col": "raw_body",
            "date_col": "date",
            "msg_id_col": "message-id",
            "reply_col": "in-reply-to",
            "refs_col": "references"
        },
        "export_dir": "./output_data/funnel_exports"
    }
    with open(CONFIG_PATH, "w") as f:
        json.dump(default_config, f, indent=2)
    print(f"✅ Created default config at {CONFIG_PATH}")

else:
  print("✅ File input.json loaded")

with open(CONFIG_PATH, "r") as f:
  config = json.load(f)


# Expose variables exactly as the rest of the notebook expects them
BASE_DIR          = config["base_dir"]
LIST_NAME         = config["list_name"]
ANALYSIS_YEARS    = list(range(config["analysis_years"]["start"], config["analysis_years"]["end"]))
RETENTION_MONTHS  = config["retention_months"]
PATCH_SUBJECT_RE  = config["regex"]["patch_subject"]
REVIEW_BODY_RE    = config["regex"]["review_body"]
EXPORT_DIR        = config["export_dir"]

# Column names (used in the normalisation cell)
FROM_COL     = config["columns"]["from_col"]
SUBJECT_COL  = config["columns"]["subject_col"]
BODY_COL     = config["columns"]["body_col"]
DATE_COL     = config["columns"]["date_col"]
MSG_ID_COL   = config["columns"]["msg_id_col"]
REPLY_COL    = config["columns"]["reply_col"]
REFS_COL     = config["columns"]["refs_col"]

print(f"Target list : {LIST_NAME}")
print(f"Year range  : {ANALYSIS_YEARS[0]}-{ANALYSIS_YEARS[-1]}")
print(f"Retention months: {RETENTION_MONTHS}")


## 2. Data Loading

In [ ]:
df_raw = (
    spark.read.parquet(f"{BASE_DIR}/list={LIST_NAME}")
    .withColumn("list", F.lit(LIST_NAME))
)

print("=== Schema ===")
df_raw.printSchema()
df_raw.show(3, truncate=80)

In [ ]:
# Column names are loaded from input.json (see Setup cell)

select_cols = [
    F.col(FROM_COL).alias("contributor_hash"),
    F.col(DATE_COL).alias("date"),
    F.col(SUBJECT_COL).alias("subject"),
    F.col(MSG_ID_COL).alias("message_id"),
    F.col(REPLY_COL).alias("in_reply_to"),

    # --- Thread ID resolution ---
    # `references` array lists ancestors oldest→newest; element [0] is the
    # root of the thread.  Fall back to `in-reply-to` when references is
    # absent/empty, and finally to the message's own id for root messages.
    F.when(
        F.size(F.col(REFS_COL)) > 0,
        F.col(REFS_COL).getItem(0)
    ).when(
        F.col(REPLY_COL).isNotNull(),
        F.col(REPLY_COL)
    ).otherwise(
        F.col(MSG_ID_COL)
    ).alias("thread_id"),
]
if BODY_COL:
    select_cols.append(F.col(BODY_COL).alias("body"))

df = (
    df_raw.select(select_cols)
    .filter(F.year("date").isin(ANALYSIS_YEARS))
    .cache()
)

total      = df.count()
n_threads  = df.select("thread_id").distinct().count()
n_roots    = df.filter(F.col("thread_id") == F.col("message_id")).count()

print(f"Total messages in analysis window : {total:,}")
print(f"Distinct threads                  : {n_threads:,}")
print(f"Thread-root messages              : {n_roots:,}")
print(f"Avg messages per thread           : {total / n_threads:.1f}")

print("\n\n=== Schema ===")
df.printSchema()
df.show(3, truncate=80)

## 3. Stage 1 — New Contributor Detection

A contributor is **new** on the date of their very first message to this list.
Each unique `contributor_hash` = one person.

In [ ]:
# First message date per contributor
first_message = (
    df
    .groupBy("contributor_hash")
    .agg(
        F.min("date").alias("first_message_date"),
        F.count("*").alias("total_messages")
    )
    .withColumn("entry_year", F.year("first_message_date"))
    .withColumn("entry_month", F.trunc("first_message_date", "month"))
)

print(f"Unique contributors: {first_message.count():,}")
first_message.orderBy("first_message_date").show(5)
print(f"top contributors")
first_message.orderBy("total_messages").show(5)

print("=== Schema ===")
first_message.printSchema()
first_message.show(3, truncate=80)

## 4. Stage 2 — First Patch Detection

A message is a **patch** when its subject matches `[PATCH` (case-insensitive).
We record the date of each contributor's first such message.

In [ ]:
# Flag patch messages
df_patches = (
    df
    .filter(F.col("subject").rlike(PATCH_SUBJECT_RE))
    .groupBy("contributor_hash")
    .agg(F.min("date").alias("first_patch_date"))
)

print(f"Contributors who submitted ≥1 patch: {df_patches.count():,}")
df_patches.show(5)

print("=== Schema ===")
df_patches.printSchema()

## 5. Stage 3 — First Review Tag Detection

A contributor has been **reviewed** when any message in the thread contains
`Reviewed-by:` or `Acked-by:` and the recipient is that contributor.

> **Note:** If your dataset stores trailer lines in a structured column
> (e.g. `reviewed_by`, `acked_by`), prefer that over the body regex in the input.json file.
> The regex fallback works on raw `body` text.

In [ ]:
if BODY_COL:
    # Extract contributor hashes mentioned in review tags from the body
    # This flags the *message* that carries a review tag — join back to who sent the patch
    df_reviewed_msgs = (
        df
        .filter(F.col("body").rlike(REVIEW_BODY_RE))
        .select("message_id", "date", "contributor_hash")
        .withColumnRenamed("contributor_hash", "reviewer_hash")
    )

    # Check for the existence of in_reply_to and thread_id columns
    if "in_reply_to" in df.columns and "thread_id" in df.columns:

        # Join reviewed messages back to the patch author via thread linkage.
        df_thread_map = df.select("message_id", "contributor_hash", "in_reply_to")

        # --- Ambiguity Resolution with Alias ---
        # Assign short names ("rev" and "map") to the column references
        df_rev = df_reviewed_msgs.alias("rev")
        df_map = df_thread_map.alias("map")


        df_first_review = (
            df_rev
            .join(
                df_map,
                F.col("rev.message_id") == F.col("map.in_reply_to"),
                "inner"
            )
            .groupBy(F.col("map.contributor_hash").alias("contributor_hash"))
            .agg(F.min(F.col("rev.date")).alias("first_review_date"))
        )

    else:
        # --- Flag any contributor who sent a reviewed msg
        # Conservative proxy when thread linkage is unavailable
        df_first_review = (
            df_reviewed_msgs
            .groupBy("reviewer_hash")
            .agg(F.min("date").alias("first_review_date"))
            .withColumnRenamed("reviewer_hash", "contributor_hash")
        )

else:
    # Body not available — create an empty placeholder
    from pyspark.sql.types import StructType, StructField, StringType, TimestampType
    schema = StructType([
        StructField("contributor_hash", StringType()),
        StructField("first_review_date", TimestampType())
    ])
    df_first_review = spark.createDataFrame([], schema)
    print("[WARN] body column not available — review stage skipped.")

print(f"Contributors with ≥1 review tag: {df_first_review.count():,}")
print("=== Schema ===")
df_first_review.printSchema()

## 6. Stage 4 — Retention Analysis

A contributor is **retained** at checkpoint *k* months if they posted
at least one message between `first_message_date + k months` and
`first_message_date + k months + 30 days`.

In [ ]:
def compute_retention(df_all, first_msg_df, months):
    """
    Returns a DataFrame with columns:
      contributor_hash, first_message_date, retained_{months}m (bool)
    """
    # Window: [first_message_date + months, first_message_date + months + 30d]
    window_start = F.add_months("first_message_date", months)
    window_end   = F.date_add(window_start, 30)

    # All messages per contributor that fall inside the retention window
    activity_in_window = (
        df_all
        .join(first_msg_df.select("contributor_hash", "first_message_date"),
              on="contributor_hash", how="inner")
        .filter(
            (F.col("date") >= window_start) &
            (F.col("date") <= window_end)
        )
        .groupBy("contributor_hash")
        .agg(F.count("*").alias(f"msgs_at_{months}m"))
        .withColumn(f"retained_{months}m", F.lit(True))
        .select("contributor_hash", f"retained_{months}m")
    )
    return activity_in_window


# Compute retention for each checkpoint
retention_dfs = [
    compute_retention(df, first_message, m) for m in RETENTION_MONTHS
]

# Report
for m, rdf in zip(RETENTION_MONTHS, retention_dfs):
    n = rdf.count()
    total = first_message.count()
    print(f"Retained at {m:2d} months: {n:,} / {total:,}  ({100*n/total:.1f}%)")

print("=== Schema ===")
first_message.printSchema()

## 7. Derived Metrics

Assemble the full funnel table and compute the three key metrics.

In [ ]:
 # ── 7a. Build the funnel table ────────────────────────────────────────────────
funnel = first_message  # start with all contributors

# Join patch stage
funnel = funnel.join(df_patches, on="contributor_hash", how="left")

# Join review stage
funnel = funnel.join(df_first_review, on="contributor_hash", how="left")

# Join retention stages
for m, rdf in zip(RETENTION_MONTHS, retention_dfs):
    funnel = funnel.join(rdf, on="contributor_hash", how="left")
    funnel = funnel.withColumn(
        f"retained_{m}m",
        F.coalesce(F.col(f"retained_{m}m"), F.lit(False))
    )

# Stage flags
funnel = (
    funnel
    .withColumn("reached_patch",  F.col("first_patch_date").isNotNull())
    .withColumn("reached_review", F.col("first_review_date").isNotNull())
    # Days from first message to first patch
    .withColumn(
        "days_to_first_patch",
        F.datediff("first_patch_date", "first_message_date")
    )
    .cache()
)

print("Funnel table schema:")
funnel.printSchema()

In [ ]:
# ── 7b. Metric 1: Mean Time to First Patch ───────────────────────────────────
mttfp = (
    funnel
    .filter(F.col("reached_patch") & (F.col("days_to_first_patch") >= 0))
    .agg(
        F.mean("days_to_first_patch").alias("mean_days"),
        F.expr("percentile(days_to_first_patch, 0.5)").alias("median_days"),
        F.stddev("days_to_first_patch").alias("std_days")
    )
    .toPandas()
)

print("=== Metric 1: Mean Time to First Patch ===")
print(f"  Mean   : {mttfp['mean_days'][0]:.1f} days")
print(f"  Median : {mttfp['median_days'][0]:.1f} days")
print(f"  Std    : {mttfp['std_days'][0]:.1f} days")

In [ ]:

# ── 7c. Metric 2: Review Conversion Rate ─────────────────────────────────────
review_conv = (
    funnel
    .filter(F.col("reached_patch"))     # only count those who submitted a patch
    .agg(
        F.count("*").alias("total_patchers"),
        F.sum(F.col("reached_review").cast("int")).alias("reviewed_count")
    )
    .withColumn("review_conversion_pct", F.round(
        100 * F.col("reviewed_count") / F.col("total_patchers"), 2)
    )
    .toPandas()
)

print("=== Metric 2: Review Conversion Rate ===")
print(review_conv.to_string(index=False))

In [ ]:
# ── 7d. Metric 3: Cohort Retention by Entry Year ─────────────────────────────
retention_cols = [f"retained_{m}m" for m in RETENTION_MONTHS]

cohort_retention = (
    funnel
    .groupBy("entry_year")
    .agg(
        F.count("*").alias("cohort_size"),
        F.sum(F.col("reached_patch").cast("int")).alias("n_patched"),
        F.sum(F.col("reached_review").cast("int")).alias("n_reviewed"),
        *[F.sum(F.col(c).cast("int")).alias(f"n_{c}") for c in retention_cols]
    )
    .orderBy("entry_year")
)

# Compute conversion rates at each stage
for stage, col in [("patch", "n_patched"), ("review", "n_reviewed")] + \
                   [(f"ret_{m}m", f"n_retained_{m}m") for m in RETENTION_MONTHS]:
    cohort_retention = cohort_retention.withColumn(
        f"pct_{stage}",
        F.round(100 * F.col(col) / F.col("cohort_size"), 2)
    )

pdf_cohort = cohort_retention.toPandas()

print("=== Metric 3: Cohort Retention by Entry Year ===")
print(pdf_cohort[["entry_year", "cohort_size",
                   "pct_patch", "pct_review",
                   *[f"pct_ret_{m}m" for m in RETENTION_MONTHS]]].to_string(index=False))

## 8. Export for Visualization

Export to JSON/CSV so the interactive HTML dashboard can consume these tables directly.

In [ ]:
import json

EXPORT_DIR = config.get("export_dir", "./output_data/funnel_exports")
import os; os.makedirs(EXPORT_DIR, exist_ok=True)

# ── Overall funnel summary (for the Sankey / funnel bar) ─────────────────────
total = funnel.count()
n_patched  = funnel.filter("reached_patch").count()
n_reviewed = funnel.filter("reached_review").count()
n_ret6     = funnel.filter(f"retained_6m").count()  if 6  in RETENTION_MONTHS else 0
n_ret12    = funnel.filter(f"retained_12m").count() if 12 in RETENTION_MONTHS else 0

funnel_summary = {
    "list": LIST_NAME,
    "stages": [
        {"label": "New Contributor", "n": total,      "pct": 100.0},
        {"label": "First Patch",     "n": n_patched,  "pct": round(100*n_patched/total, 2)},
        {"label": "First Review",    "n": n_reviewed, "pct": round(100*n_reviewed/total, 2)},
        {"label": "Retained 6m",     "n": n_ret6,     "pct": round(100*n_ret6/total, 2)},
        {"label": "Retained 12m",    "n": n_ret12,    "pct": round(100*n_ret12/total, 2)},
    ],
    "mttfp_days": float(mttfp["mean_days"][0]),
    "review_conversion_pct": float(review_conv["review_conversion_pct"][0]),
}

with open(f"{EXPORT_DIR}/funnel_summary.json", "w") as f:
    json.dump(funnel_summary, f, indent=2)

# ── Cohort table (for the temporal slider view) ───────────────────────────────
pdf_cohort.to_csv(f"{EXPORT_DIR}/cohort_retention.csv", index=False)

print("Exported:")
print(f"  {EXPORT_DIR}/funnel_summary.json")
print(f"  {EXPORT_DIR}/cohort_retention.csv")
print(json.dumps(funnel_summary, indent=2))

## 9. Download Results to Your Computer

> After the analysis finishes, run the cell below to **automatically download** all files directly to your computer.

- Downloads the configuration file **`input.json`**
- Zips the entire export folder (`funnel_exports/`) into a single file
- Downloads **`funnel_exports.zip`** which contains:
  - `funnel_summary.json` (overall funnel metrics)
  - `cohort_retention.csv` (year-by-year retention table)

Both files will be saved automatically in your **Downloads** folder.

In [ ]:
from google.colab import files
import shutil
import os

print("📥 Preparing files for download...")

# 1. Download input.json (configuration file)
if os.path.exists("/content/drive/MyDrive/input.json"):
    files.download("/content/drive/MyDrive/input.json")
    print("✅ Downloaded: input.json")

# 2. Zip and download the entire exports folder (funnel_summary.json + cohort_retention.csv)
if os.path.exists(EXPORT_DIR):
    zip_path = "/content/funnel_exports.zip"
    shutil.make_archive(zip_path.replace(".zip", ""), 'zip', EXPORT_DIR)
    files.download(zip_path)
    print("✅ Downloaded: funnel_exports.zip (contains both result files)")
else:
    print("⚠️ Export folder not found")